# Лабораторная 3: фактическая локальная baseline

Модель: Qwen 2.5 3B / Ollama, решение ADR-006. Нет RAG, поэтому метрик поиска нет.
Это измерение учебных черновиков, не времени до публикации. Воспроизводимый
live-прогон выполняется следующей командой из корня репозитория; он перезаписывает
артефакты, поэтому запускайте его осознанно:

```sh
python -m ml.experiment --dataset docs/golden-dataset/v1.jsonl --split dev --limit 6 --repeats 1 --generate --output docs/experiments/lab3-run
```

Ячейки ниже анализируют сохранённые JSONL/CSV и не маскируют повторный запуск чтением oracle-ответов.
Для полного eval-набора см. ветку `feature/quality_lab3`, `tests/evals/README.md`.


In [1]:
from pathlib import Path
import json, csv, statistics
root=Path.cwd()
while not (root / "ml/models/baseline.py").exists():
    if root.parent == root: raise RuntimeError("Open notebook inside the repository")
    root=root.parent
artifact=root / "docs/experiments/lab3-run"
run=json.loads((artifact / "run.json").read_text(encoding="utf-8"))
rows=[json.loads(s) for s in (artifact / "requests.jsonl").read_text(encoding="utf-8").splitlines() if s.strip()]
print({k:run[k] for k in ("model","provider","records","repeats","split","dataset_sha256")})


{'model': 'qwen2.5:3b', 'provider': 'ollama', 'records': 6, 'repeats': 1, 'split': 'dev', 'dataset_sha256': 'ca18ebfff1db6c9f02cf0a8203abb7b8a645f86e8a785a518019616819a635a0'}


In [2]:
def percentile(xs,q):
    xs=sorted(xs); at=(len(xs)-1)*q; left=int(at)
    return xs[left]+(xs[min(left+1,len(xs)-1)]-xs[left])*(at-left)
latencies=[r["latency_ms"]/1000 for r in rows]
usage=[r["usage"] for r in rows if r.get("usage")]
summary={"requests":len(rows), "successes":sum(r["status"]=="ok" for r in rows),
         "latency_p50_seconds":percentile(latencies,.5),
         "latency_p95_seconds":percentile(latencies,.95),
         "input_tokens":sum(u["input_tokens"] for u in usage),
         "output_tokens":sum(u["output_tokens"] for u in usage),
         "external_api_token_fee_rub":sum(u["cost_rub"] for u in usage)}
summary


{'requests': 6,
 'successes': 6,
 'latency_p50_seconds': 11.004912650001643,
 'latency_p95_seconds': 14.114347600007022,
 'input_tokens': 9116,
 'output_tokens': 3075,
 'external_api_token_fee_rub': 0.0}

In [3]:
with (artifact / "usage.csv").open(encoding="utf-8",newline="") as f:
    exported=list(csv.DictReader(f))
assert len(exported)==len(rows)
[(r["id"],r["status"],round(r["latency_ms"]/1000,2),
  (r.get("usage") or {}).get("input_tokens"),
  (r.get("usage") or {}).get("output_tokens")) for r in rows]


[('gd-0001', 'ok', 14.45, 1657, 498),
 ('gd-0002', 'ok', 8.18, 1479, 425),
 ('gd-0003', 'ok', 13.11, 1506, 673),
 ('gd-0004', 'ok', 8.05, 1407, 399),
 ('gd-0005', 'ok', 11.96, 1581, 595),
 ('gd-0006', 'ok', 10.05, 1486, 485)]

## Интерпретация для продукта

Смотрите фактическую записку `lab3.md`: квантили latency относятся к указанной
малой выборке и оборудованию, без очереди нескольких пользователей. Нулевой
API-тариф не равен нулевой полной стоимости. Время публикации с человеком не измерено.
Схема JSON доказывает только формат: выдумки, полнота адреса и календарь оцениваются
отдельно. Не включаем дополнительные факты в публикацию без подтверждения экспертом.
Следующие изменения выбираются по live-ошибкам, без изменения эталонов под модель.
